# MULTI-ARMED-BANDITS PROBLEM

### Environment Setup
Run the following in your terminal:
```
conda create -n rl-bandits python=3.11 -y
conda activate rl-bandits
```

In [1]:
# %pip install torch numpy matplotlib gymnasium ipykernel

### Dependencies

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

# Device selection
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

PyTorch version: 2.10.0
Device: mps


### Multi-Armed Bandit Environment

In [3]:
class MultiArmedBandit:
    def __init__(self, k=10, seed=None):
        self.k = k
        self.gen = torch.Generator()
        if seed is not None:
            self.gen.manual_seed(seed)
        self.true_values = None
        self.optimal_action = None

    def reset(self):
        self.true_values = torch.normal(0.0, 1.0, size=(self.k,), generator=self.gen)
        self.optimal_action = torch.argmax(self.true_values).item()
        return 0, {}

    def step(self, action):
        reward = torch.normal(self.true_values[action], 1.0, size=(1,), generator=self.gen).item()
        return 0, reward, False, False, {}

### Epsilon-Greedy Agent

In [4]:
class EpsilonGreedyAgent:
    def __init__(self, k=10, epsilon=0.1):
        self.k = k
        self.epsilon = epsilon
        self.Q = torch.zeros(k)
        self.N = torch.zeros(k)

    def select_action(self):
        if torch.rand(1).item() < self.epsilon:
            return torch.randint(0, self.k, (1,)).item()
        else:
            return torch.argmax(self.Q).item()

    def update(self, action, reward):
        self.N[action] += 1
        self.Q[action] += (1.0 / self.N[action]) * (reward - self.Q[action])

### UCB Agent

In [5]:
class UCBAgent:
    def __init__(self, k=10, c=2.0):
        self.k = k
        self.c = c
        self.Q = torch.zeros(k)
        self.N = torch.zeros(k)
        self.t = 0

    def select_action(self):
        self.t += 1
        if self.t <= self.k:
            return self.t - 1  # warm-up: pull each arm once
        ucb_values = self.Q + self.c * torch.sqrt(torch.log(torch.tensor(self.t, dtype=torch.float32)) / self.N)
        return torch.argmax(ucb_values).item()

    def update(self, action, reward):
        self.N[action] += 1
        self.Q[action] += (1.0 / self.N[action]) * (reward - self.Q[action])

### Experiment Loop

In [6]:
def run_experiment(agent_configs, n_runs=1000, n_steps=2000, k=10):
    """
    Run all agents on bandits with the same true values but independent reward noise.
    
    agent_configs: list of (name, agent_class, kwargs) tuples
    Returns: dict of {name: {'rewards': tensor, 'optimal': tensor}}
             where each tensor is shape (n_steps,) averaged over all runs
    """
    n_agents = len(agent_configs)
    all_rewards = torch.zeros(n_agents, n_steps)
    all_optimal = torch.zeros(n_agents, n_steps)

    for run in range(n_runs):
        if run % 100 == 0:
            print(f"Run {run}/{n_runs}...")

        # Each agent gets its own bandit (same seed = same true means, independent noise)
        bandits = [MultiArmedBandit(k=k, seed=run + 1) for _ in agent_configs]
        for b in bandits:
            b.reset()

        agents = [cls(k=k, **kwargs) for _, cls, kwargs in agent_configs]

        for step in range(n_steps):
            for i, (agent, bandit) in enumerate(zip(agents, bandits)):
                action = agent.select_action()
                _, reward, _, _, _ = bandit.step(action)
                agent.update(action, reward)

                all_rewards[i, step] += reward
                all_optimal[i, step] += (action == bandit.optimal_action)

    # Average over runs
    all_rewards /= n_runs
    all_optimal /= n_runs

    results = {}
    for i, (name, _, _) in enumerate(agent_configs):
        results[name] = {
            'rewards': all_rewards[i],
            'optimal': all_optimal[i]
        }
    print("Done!")
    return results

### Run Experiment

In [7]:
agent_configs = [
    ("ε=0.01", EpsilonGreedyAgent, {"epsilon": 0.01}),
    ("ε=0.1",  EpsilonGreedyAgent, {"epsilon": 0.1}),
    ("ε=0.2",  EpsilonGreedyAgent, {"epsilon": 0.2}),
    ("UCB c=1", UCBAgent, {"c": 1.0}),
    ("UCB c=2", UCBAgent, {"c": 2.0}),
]

results = run_experiment(agent_configs)
print("Experiment complete.")

Run 0/1000...
Run 100/1000...
Run 200/1000...
Run 300/1000...
Run 400/1000...
Run 500/1000...
Run 600/1000...
Run 700/1000...
Run 800/1000...
Run 900/1000...
Done!
Experiment complete.


### Results

In [ ]:
# Plot and save results
steps = torch.arange(1, 2001)

# --- Average Reward ---
fig1, ax1 = plt.subplots(figsize=(10, 5))
for name, data in results.items():
    ax1.plot(steps.numpy(), data['rewards'].numpy(), label=name)
ax1.set_xlabel('Steps')
ax1.set_ylabel('Average Reward')
ax1.set_title('Average Reward vs. Steps')
ax1.legend()
fig1.tight_layout()
fig1.savefig('avg_reward.png', dpi=150)
print("Saved avg_reward.png")
plt.show()

# --- % Optimal Action ---
fig2, ax2 = plt.subplots(figsize=(10, 5))
for name, data in results.items():
    ax2.plot(steps.numpy(), data['optimal'].numpy() * 100, label=name)
ax2.set_xlabel('Steps')
ax2.set_ylabel('% Optimal Action')
ax2.set_title('% Optimal Action vs. Steps')
ax2.legend()
fig2.tight_layout()
fig2.savefig('optimal_action.png', dpi=150)
print("Saved optimal_action.png")
plt.show()

In [9]:
# Quantitative Summary
print("=" * 70)
print(f"{'Strategy':<12} {'Final Reward':>14} {'Final % Optimal':>16} {'Cumul. Reward':>14}")
print("=" * 70)

for name, data in results.items():
    final_reward = data['rewards'][-1].item()
    final_optimal = data['optimal'][-1].item() * 100
    cumulative_reward = data['rewards'].sum().item()
    print(f"{name:<12} {final_reward:>14.4f} {final_optimal:>15.2f}% {cumulative_reward:>14.2f}")

print("=" * 70)

# Best performer
best_name = max(results, key=lambda n: results[n]['optimal'][-1].item())
print(f"\nBest strategy by % optimal action at step 2000: {best_name}")
print(f"  Final % optimal: {results[best_name]['optimal'][-1].item() * 100:.2f}%")
print(f"  Final avg reward: {results[best_name]['rewards'][-1].item():.4f}")
print(f"  Total cumulative reward: {results[best_name]['rewards'].sum().item():.2f}")

# Early performance (first 100 steps)
print(f"\nEarly learning (avg reward over first 100 steps):")
for name, data in results.items():
    early_avg = data['rewards'][:100].mean().item()
    print(f"  {name:<12} {early_avg:.4f}")

# Late performance (last 100 steps)
print(f"\nConverged performance (avg reward over last 100 steps):")
for name, data in results.items():
    late_avg = data['rewards'][-100:].mean().item()
    print(f"  {name:<12} {late_avg:.4f}")

Strategy       Final Reward  Final % Optimal  Cumul. Reward
ε=0.01               1.3734           69.10%        2515.51
ε=0.1                1.3581           82.30%        2618.16
ε=0.2                1.1829           74.60%        2356.68
UCB c=1              1.5000           94.70%        2947.83
UCB c=2              1.4736           89.80%        2831.93

Best strategy by % optimal action at step 2000: UCB c=1
  Final % optimal: 94.70%
  Final avg reward: 1.5000
  Total cumulative reward: 2947.83

Early learning (avg reward over first 100 steps):
  ε=0.01       0.9329
  ε=0.1        0.9776
  ε=0.2        0.9323
  UCB c=1      1.1664
  UCB c=2      0.9327

Converged performance (avg reward over last 100 steps):
  ε=0.01       1.3788
  ε=0.1        1.3449
  ε=0.2        1.1965
  UCB c=1      1.5029
  UCB c=2      1.4813
